# Snippet from Math-Boundary-Gap-and-Entropy.md


In [ ]:
import numpy as np
from scipy.special import softmax, entr

def boundary_flag(
    utilities: np.ndarray,
    unc_winner: float,
    delta_gap: float = 0.05,
    delta_h: float = 1.0,
    tau: float = 1.0
) -> dict:
    """
    Detect boundaries using gap and entropy.
    
    Args:
        utilities: Descending sorted utilities.
        unc_winner: Top candidate uncertainty.
        delta_gap: Gap limit (default 0.05).
        delta_h: Entropy limit in nats (default 1.0).
        tau: Softmax temperature (default 1.0).
    
    Returns:
        Dict with gap, entropy, flag, reason.
    """
    gap = utilities[0] - utilities[1] if len(utilities) > 1 else float('inf')
    p = softmax(tau * utilities)
    H = np.sum(entr(p))
    is_boundary = (gap < delta_gap) and (H > delta_h or unc_winner > 0.1)
    reason = "clear"
    if is_boundary:
        if H > delta_h and unc_winner > 0.1:
            reason = "high_entropy_high_uncertainty"
        elif H > delta_h:
            reason = "high_entropy"
        else:
            reason = "high_uncertainty"
    return {
        "gap": float(gap),
        "entropy": float(H),
        "unc_winner": float(unc_winner),
        "flag": bool(is_boundary),
        "reason": reason,
        "metadata": {
            "tau": tau,
            "delta_gap": delta_gap,
            "delta_h": delta_h,
            "n_candidates": len(utilities)
        }
    }

# Examples
if __name__ == "__main__":
    U_tie = np.array([0.80, 0.75, 0.60])
    print("Tie:", boundary_flag(U_tie, unc_winner=0.05))
    # Output: {'gap': 0.05, 'entropy': 1.02, 'unc_winner': 0.05, 'flag': True, 'reason': 'high_entropy', ...}
    
    U_clear = np.array([0.95, 0.60, 0.50])
    print("Clear:", boundary_flag(U_clear, unc_winner=0.02))
    # Output: {'gap': 0.35, 'entropy': 0.41, 'unc_winner': 0.02, 'flag': False, 'reason': 'clear', ...}
    
    U_uncertain = np.array([0.85, 0.80, 0.65])
    print("Uncertain:", boundary_flag(U_uncertain, unc_winner=0.15))
    # Output: {'gap': 0.05, 'entropy': 0.95, 'unc_winner': 0.15, 'flag': True, 'reason': 'high_uncertainty', ...}
